<a href="https://colab.research.google.com/github/carlosparissouza-sketch/Carlos-12/blob/main/Farma_tech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
FARMA TECH - Capanema/PR
Aplicativo Flask + SQLite

Recursos:
- Cadastro público de usuário comum e pessoa pública
- Login com senha protegida por hash
- Administrador inicial configurável
- Cadastro, edição e exclusão de farmácias somente pelo administrador
- Aprovação de perfis de pessoas públicas pelo administrador
- Favoritos para usuários autenticados
- Busca de farmácias

Execute:
    pip install flask werkzeug
    python app.py

Acesse: http://127.0.0.1:5000
Administrador inicial: admin / 123456
Troque essas credenciais antes de usar em produção.
"""

import os
import secrets
import sqlite3
import urllib.parse
import requests
from contextlib import closing
from functools import wraps
from pathlib import Path

from flask import (
    Flask, flash, g, redirect, render_template_string,
    request, session, url_for
)
from werkzeug.security import check_password_hash, generate_password_hash


# -----------------------------------------------------------------------------
# Configuração
# -----------------------------------------------------------------------------

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    # No Google Colab, __file__ não existe.
    BASE_DIR = Path("/content")
DB_PATH = Path(os.environ.get("FARMATECH_DB", BASE_DIR / "farmatech.db"))
ADMIN_USUARIO = os.environ.get("FARMATECH_ADMIN", "admin")
ADMIN_SENHA = os.environ.get("FARMATECH_ADMIN_SENHA", "123456")
ADMIN_EMAIL = os.environ.get("FARMATECH_ADMIN_EMAIL", "carlos.paris.souza@escola.pr.gov.br")
GOOGLE_CLIENT_ID = os.environ.get("GOOGLE_CLIENT_ID", "")
GOOGLE_CLIENT_SECRET = os.environ.get("GOOGLE_CLIENT_SECRET", "")
GOOGLE_REDIRECT_URI = os.environ.get("GOOGLE_REDIRECT_URI", "")

app = Flask(__name__)

app.config.update(
    SECRET_KEY=os.environ.get(
        "FARMATECH_SECRET",
        "troque-esta-chave-secreta-antes-de-publicar",
    ),
    DATABASE=str(DB_PATH),
)


# -----------------------------------------------------------------------------
# Banco de dados
# -----------------------------------------------------------------------------

def get_db():
    if "db" not in g:
        g.db = sqlite3.connect(app.config["DATABASE"])
        g.db.row_factory = sqlite3.Row
        g.db.execute("PRAGMA foreign_keys = ON")
    return g.db


@app.teardown_appcontext
def close_db(_error=None):
    db = g.pop("db", None)
    if db is not None:
        db.close()


def init_db():
    db = get_db()
    db.executescript(
        """
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            usuario TEXT UNIQUE NOT NULL,
            email TEXT,
            telefone TEXT,
            senha TEXT NOT NULL,
            tipo TEXT NOT NULL DEFAULT 'usuario'
                CHECK (tipo IN ('usuario', 'pessoa_publica', 'admin')),
            ativo INTEGER NOT NULL DEFAULT 1,
            criado_em TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        CREATE TABLE IF NOT EXISTS pessoas_publicas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            usuario_id INTEGER NOT NULL UNIQUE,
            nome_completo TEXT NOT NULL,
            nome_publico TEXT,
            documento TEXT,
            email TEXT,
            telefone TEXT,
            cidade TEXT,
            estado TEXT,
            cargo TEXT,
            orgao TEXT,
            biografia TEXT,
            foto TEXT,
            redes_sociais TEXT,
            site_oficial TEXT,
            status TEXT NOT NULL DEFAULT 'Pendente'
                CHECK (status IN ('Pendente', 'Aprovado', 'Rejeitado')),
            criado_em TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (usuario_id) REFERENCES usuarios(id) ON DELETE CASCADE
        );

        CREATE TABLE IF NOT EXISTS farmacias (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            categoria TEXT NOT NULL DEFAULT 'Farmácia',
            endereco TEXT,
            bairro TEXT,
            telefone TEXT,
            horarios TEXT,
            avaliacao REAL NOT NULL DEFAULT 0,
            entrega TEXT DEFAULT 'Não informado',
            destaque TEXT DEFAULT 'Não',
            latitude TEXT,
            longitude TEXT,
            descricao TEXT,
            criado_em TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        CREATE TABLE IF NOT EXISTS favoritos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            usuario_id INTEGER NOT NULL,
            farmacia_id INTEGER NOT NULL,
            UNIQUE (usuario_id, farmacia_id),
            FOREIGN KEY (usuario_id) REFERENCES usuarios(id) ON DELETE CASCADE,
            FOREIGN KEY (farmacia_id) REFERENCES farmacias(id) ON DELETE CASCADE
        );
        """
    )
    admin = db.execute(
        "SELECT id FROM usuarios WHERE usuario = ?", (ADMIN_USUARIO,)
    ).fetchone()
    if admin is None:
        db.execute(
            """
            INSERT INTO usuarios (nome, usuario, email, senha, tipo, ativo)
            VALUES (?, ?, ?, ?, 'admin', 1)
            """,
            (
                "Administrador",
                ADMIN_USUARIO,
                "admin@farmatech.local",
                generate_password_hash(ADMIN_SENHA),
            ),
        )
    # Promove a conta local já cadastrada com o e-mail autorizado.
    # Isto não acessa o Gmail nem altera a senha da conta.
    db.execute(
        "UPDATE usuarios SET tipo = 'admin', ativo = 1 WHERE lower(email) = lower(?)",
        (ADMIN_EMAIL,),
    )
    db.commit()


# -----------------------------------------------------------------------------
# Autenticação e autorização
# -----------------------------------------------------------------------------

def current_user():
    user_id = session.get("usuario_id")
    if not user_id:
        return None
    return get_db().execute(
        "SELECT * FROM usuarios WHERE id = ? AND ativo = 1", (user_id,)
    ).fetchone()


def login_required(view):
    @wraps(view)
    def wrapped(*args, **kwargs):
        if current_user() is None:
            flash("Faça login para continuar.", "warning")
            return redirect(url_for("login", next=request.path))
        return view(*args, **kwargs)
    return wrapped


def admin_required(view):
    @wraps(view)
    def wrapped(*args, **kwargs):
        user = current_user()
        if user is None or user["tipo"] != "admin":
            flash("Acesso restrito ao administrador.", "danger")
            return redirect(url_for("login"))
        return view(*args, **kwargs)
    return wrapped


@app.context_processor
def inject_globals():
    return {"usuario_logado": current_user()}


# -----------------------------------------------------------------------------
# Layout e estilos
# -----------------------------------------------------------------------------

STYLE = """
<style>
:root { --green:#11853e; --dark:#123322; --soft:#f2faf5; --line:#dbe9df; }
* { box-sizing:border-box; }
body { margin:0; font-family:Arial,Helvetica,sans-serif; background:var(--soft); color:#183126; }
nav { background:white; border-bottom:1px solid var(--line); padding:14px 6%; display:flex; align-items:center; justify-content:space-between; gap:20px; flex-wrap:wrap; }
.logo { color:var(--dark); text-decoration:none; font-size:21px; font-weight:bold; display:flex; align-items:center; gap:9px; }
.logo i { background:#18a04b; color:#fff; border-radius:10px; font-style:normal; font-size:27px; width:40px; height:40px; display:grid; place-items:center; }
.logo span { color:#0b9d43; }
.menu { display:flex; gap:15px; align-items:center; flex-wrap:wrap; }
a { color:var(--green); font-weight:bold; text-decoration:none; }
.menu a { color:#53675b; font-weight:normal; }
.container { width:88%; max-width:1160px; margin:34px auto; }
.hero { background:linear-gradient(110deg,#105d2f,#16a34a); color:#fff; padding:66px 7% 75px; }
.hero h1 { max-width:790px; font-size:clamp(38px,6vw,64px); line-height:1.02; letter-spacing:-2px; margin:12px 0; }
.hero h1 span { color:#baf6cd; }
.hero p { color:#dcf7e4; font-size:18px; }
.badge { display:inline-block; background:#e5f8eb; color:#087738; border-radius:30px; padding:9px 15px; font-size:13px; font-weight:bold; }
.search { display:flex; max-width:730px; background:#fff; border-radius:14px; padding:7px; margin-top:26px; }
.search input { flex:1; min-width:0; border:0; outline:0; padding:16px; font-size:16px; }
.search button { border:0; border-radius:10px; background:#0d9841; color:#fff; font-weight:bold; padding:0 24px; cursor:pointer; }
.grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(270px,1fr)); gap:18px; }
.card, .panel { background:#fff; border:1px solid var(--line); border-radius:16px; padding:22px; box-shadow:0 4px 14px #1233220b; }
.card h3 { margin:7px 0 13px; }
.category { text-transform:uppercase; color:#079342; font-size:12px; font-weight:bold; }
.info { color:#64766b; font-size:14px; line-height:1.8; }
.star { color:#e29a00; font-weight:bold; } .delivery { color:#078d3d; font-weight:bold; } .highlight { color:#b47b00; font-weight:bold; }
.description { color:#52675a; line-height:1.6; margin-top:10px; }
.btn, button.btn { display:inline-block; border:0; border-radius:9px; background:#12a349; color:#fff; padding:11px 16px; font-weight:bold; cursor:pointer; }
.btn.secondary { background:#e8f5ed; color:#087c39; } .btn.danger { background:#c93636; }
.actions { display:flex; gap:8px; flex-wrap:wrap; margin-top:14px; }
.form-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:14px; }
.field { display:flex; flex-direction:column; gap:6px; } .full { grid-column:1/-1; }
input, select, textarea { width:100%; border:1px solid #cfdfd4; border-radius:8px; padding:12px; font:inherit; background:#fff; }
textarea { min-height:95px; resize:vertical; }
label { font-weight:bold; font-size:14px; } small, .muted { color:#6b7d71; }
.flash { padding:12px 15px; border-radius:9px; margin-bottom:13px; background:#e6f5eb; color:#176b38; } .flash.danger { background:#ffe8e8; color:#9c2929; } .flash.warning { background:#fff4d7; color:#805900; }
.table-wrap { overflow:auto; } table { border-collapse:collapse; width:100%; background:#fff; } th,td { text-align:left; padding:11px; border-bottom:1px solid var(--line); vertical-align:top; }
.empty { text-align:center; background:#fff; padding:36px; border-radius:16px; }.tabs { display:flex; gap:8px; flex-wrap:wrap; margin:0 0 18px; } .tab { border:1px solid #cfe2d5; background:#fff; color:#087c39; border-radius:9px; padding:11px 16px; font-weight:bold; text-decoration:none; cursor:pointer; } .tab.active, .tab:hover { background:#12a349; color:#fff; }
@media(max-width:700px) { .hero { padding:48px 6% 58px; } .search { flex-direction:column; gap:6px; } .search button { height:48px; } .form-grid { grid-template-columns:1fr; } .full { grid-column:auto; } nav { justify-content:center; } }
</style>
"""


def page(title, body, **context):
    template = """<!doctype html><html lang='pt-BR'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>{{ title }} - Farma Tech</title>""" + STYLE + """</head><body>
<nav><a class='logo' href='{{ url_for("home") }}'><i>+</i>Farma <span>Tech</span></a><div class='menu'><a href='{{ url_for("home") }}'>Encontrar</a><a href='{{ url_for("public_people") }}'>Pessoas públicas</a><a href='{{ url_for("register") }}'>Cadastro público</a>{% if usuario_logado %}<a href='{{ url_for("favoritos") }}'>♡ Favoritos</a>{% if usuario_logado['tipo'] == 'admin' %}<a href='{{ url_for("admin") }}'>⚙ Administração</a>{% endif %}<a href='{{ url_for("logout") }}'>Sair ({{ usuario_logado['nome'] }})</a>{% else %}<a href='{{ url_for("login") }}'>Entrar</a><a href='{{ url_for("register") }}'>Cadastrar</a>{% endif %}</div></nav>
<div class='container'>{% with messages = get_flashed_messages(with_categories=true) %}{% for category,message in messages %}<div class='flash {{ category }}'>{{ message }}</div>{% endfor %}{% endwith %}{{ body|safe }}</div></body></html>"""
    return render_template_string(template, title=title, body=render_template_string(body, **context), **context)


# -----------------------------------------------------------------------------
# Área pública
# -----------------------------------------------------------------------------

@app.route("/")
def home():
    busca = request.args.get("busca", "").strip()
    db = get_db()
    if busca:
        termo = f"%{busca}%"
        pharmacies = db.execute(
            """SELECT * FROM farmacias WHERE nome LIKE ? OR endereco LIKE ? OR bairro LIKE ? OR categoria LIKE ? OR descricao LIKE ?
               ORDER BY destaque = 'Sim' DESC, avaliacao DESC, nome ASC""",
            (termo, termo, termo, termo, termo),
        ).fetchall()
    else:
        pharmacies = db.execute(
            """SELECT * FROM farmacias ORDER BY destaque = 'Sim' DESC, avaliacao DESC, nome ASC"""
        ).fetchall()
    body = """
    <section class='hero'><span class='badge'>📍 Capanema, Paraná</span><h1>Encontre uma farmácia <span>perto de você.</span></h1><p>Consulte farmácias e drogarias de Capanema em um único lugar.</p><form class='search'><input name='busca' value='{{ busca }}' placeholder='Buscar por nome, endereço, bairro...'><button>Buscar</button></form></section>
    <h2>Farmácias e drogarias</h2><p class='muted'>{{ pharmacies|length }} estabelecimento(s) encontrado(s).</p>
    {% if pharmacies %}<div class='grid'>{% for f in pharmacies %}<article class='card'><div class='category'>{{ f['categoria'] }}</div><h3>{{ f['nome'] }}</h3><div class='info'>📍 {{ f['endereco'] or 'Endereço não informado' }}<br>🏘️ {{ f['bairro'] or 'Bairro não informado' }}{% if f['telefone'] %}<br>☎️ {{ f['telefone'] }}{% endif %}<br>🕐 {{ f['horarios'] or 'Horário não informado' }}<br><span class='star'>⭐ {{ '%.1f'|format(f['avaliacao'] or 0) }}</span>{% if f['entrega'] == 'Sim' %}<br><span class='delivery'>🚚 Entrega disponível</span>{% endif %}{% if f['destaque'] == 'Sim' %}<br><span class='highlight'>⭐ Estabelecimento em destaque</span>{% endif %}</div>{% if f['descricao'] %}<div class='description'>{{ f['descricao'] }}</div>{% endif %}{% if usuario_logado %}<form class='actions' method='post' action='{{ url_for("toggle_favorite", pharmacy_id=f["id"]) }}'><button class='btn secondary'>♡ Favoritar / remover</button></form>{% endif %}</article>{% endfor %}</div>{% else %}<div class='empty'><h3>Nenhuma farmácia encontrada</h3><p>Não há estabelecimentos correspondentes à busca.</p></div>{% endif %}
    """
    return page("Início", body, busca=busca, pharmacies=pharmacies)


@app.route("/pessoas-publicas")
def public_people():
    people = get_db().execute(
        """SELECT p.* FROM pessoas_publicas p JOIN usuarios u ON u.id=p.usuario_id
           WHERE p.status='Aprovado' AND u.ativo=1 ORDER BY p.nome_publico, p.nome_completo"""
    ).fetchall()
    body = """<h1>Pessoas públicas</h1><p class='muted'>Perfis aprovados pela administração.</p>{% if people %}<div class='grid'>{% for p in people %}<article class='card'><div class='category'>{{ p['cargo'] or 'Perfil público' }}</div><h3>{{ p['nome_publico'] or p['nome_completo'] }}</h3><div class='info'>{{ p['orgao'] or '' }}{% if p['cidade'] or p['estado'] %}<br>📍 {{ p['cidade'] }}{% if p['estado'] %} - {{ p['estado'] }}{% endif %}{% endif %}</div>{% if p['biografia'] %}<p class='description'>{{ p['biografia'] }}</p>{% endif %}{% if p['site_oficial'] %}<a href='{{ p['site_oficial'] }}' target='_blank' rel='noopener'>Site oficial</a>{% endif %}</article>{% endfor %}</div>{% else %}<div class='empty'>Ainda não há perfis públicos aprovados.</div>{% endif %}"""
    return page("Pessoas públicas", body, people=people)


@app.route("/cadastro", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        form = request.form
        nome = form.get("nome", "").strip()
        usuario = form.get("usuario", "").strip().lower()
        senha = form.get("senha", "")
        tipo = form.get("tipo", "usuario")
        if not nome or not usuario or len(senha) < 6:
            flash("Informe nome, usuário e senha com pelo menos 6 caracteres.", "danger")
            return redirect(url_for("register"))
        if tipo not in ("usuario", "pessoa_publica"):
            tipo = "usuario"
        db = get_db()
        try:
            cur = db.execute(
                "INSERT INTO usuarios (nome,usuario,email,telefone,senha,tipo) VALUES (?,?,?,?,?,?)",
                (nome, usuario, form.get("email", "").strip(), form.get("telefone", "").strip(), generate_password_hash(senha), tipo),
            )
            if tipo == "pessoa_publica":
                db.execute(
                    """INSERT INTO pessoas_publicas (usuario_id,nome_completo,nome_publico,documento,email,telefone,cidade,estado,cargo,orgao,biografia,foto,redes_sociais,site_oficial)
                       VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
                    (cur.lastrowid, nome, form.get("nome_publico", "").strip(), form.get("documento", "").strip(), form.get("email", "").strip(), form.get("telefone", "").strip(), form.get("cidade", "").strip(), form.get("estado", "").strip(), form.get("cargo", "").strip(), form.get("orgao", "").strip(), form.get("biografia", "").strip(), form.get("foto", "").strip(), form.get("redes_sociais", "").strip(), form.get("site_oficial", "").strip()),
                )
            db.commit()
        except sqlite3.IntegrityError:
            db.rollback()
            flash("Este nome de usuário já está em uso.", "danger")
            return redirect(url_for("register"))
        flash("Cadastro realizado. Agora você já pode entrar.", "success")
        return redirect(url_for("login"))
    body = """<h1>Cadastro público</h1><p class='muted'>Escolha uma aba para continuar seu cadastro.</p><div class='tabs'><button type='button' class='tab active' id='tab-user' onclick="mudarTipo('usuario')">Usuário comum</button><button type='button' class='tab' id='tab-public' onclick="mudarTipo('pessoa_publica')">Pessoa Pública</button></div><form class='panel' method='post'><div class='form-grid'><div class='field'><label>Nome completo *</label><input name='nome' required></div><div class='field' style='display:none'><label>Tipo de cadastro</label><select name='tipo' id='tipo'><option value='usuario'>Usuário comum</option><option value='pessoa_publica'>Pessoa Pública</option></select></div><div class='field'><label>Usuário *</label><input name='usuario' required></div><div class='field'><label>Senha * (mínimo 6 caracteres)</label><input type='password' name='senha' minlength='6' required></div><div class='field'><label>E-mail</label><input type='email' name='email'></div><div class='field'><label>Telefone</label><input name='telefone'></div></div><div id='public-fields' hidden><hr><h2>Dados da pessoa pública</h2><div class='form-grid'><div class='field'><label>Nome público</label><input name='nome_publico'></div><div class='field'><label>CPF ou documento</label><input name='documento'></div><div class='field'><label>Cidade</label><input name='cidade'></div><div class='field'><label>Estado</label><input name='estado' maxlength='2'></div><div class='field'><label>Cargo ou função</label><input name='cargo'></div><div class='field'><label>Órgão/instituição</label><input name='orgao'></div><div class='field'><label>Foto (URL)</label><input name='foto'></div><div class='field'><label>Redes sociais</label><input name='redes_sociais'></div><div class='field'><label>Site oficial</label><input name='site_oficial' type='url'></div><div class='field full'><label>Biografia</label><textarea name='biografia'></textarea></div></div></div><button class='btn'>Criar cadastro</button></form><script>function mudarTipo(tipo){document.getElementById('tipo').value=tipo;document.getElementById('public-fields').hidden=tipo!=='pessoa_publica';document.getElementById('tab-user').classList.toggle('active',tipo==='usuario');document.getElementById('tab-public').classList.toggle('active',tipo==='pessoa_publica');}</script>"""
    return page("Cadastro", body)


@app.route("/login/google")
def google_login():
    if not GOOGLE_CLIENT_ID or not GOOGLE_CLIENT_SECRET:
        flash("O login Google ainda não foi configurado. Use usuário e senha ou configure o OAuth.", "warning")
        return redirect(url_for("login"))
    state = secrets.token_urlsafe(32)
    session["google_oauth_state"] = state
    redirect_uri = GOOGLE_REDIRECT_URI or url_for("google_callback", _external=True)
    params = {
        "client_id": GOOGLE_CLIENT_ID,
        "redirect_uri": redirect_uri,
        "response_type": "code",
        "scope": "openid email profile",
        "state": state,
        "access_type": "online",
        "prompt": "select_account",
    }
    return redirect("https://accounts.google.com/o/oauth2/v2/auth?" + urllib.parse.urlencode(params))


@app.route("/login/google/callback")
def google_callback():
    if not GOOGLE_CLIENT_ID or not GOOGLE_CLIENT_SECRET:
        flash("Login Google não configurado.", "danger")
        return redirect(url_for("login"))
    if request.args.get("state") != session.pop("google_oauth_state", None):
        flash("Sessão Google inválida ou expirada. Tente novamente.", "danger")
        return redirect(url_for("login"))
    code = request.args.get("code")
    if not code:
        flash("O login Google foi cancelado.", "warning")
        return redirect(url_for("login"))
    redirect_uri = GOOGLE_REDIRECT_URI or url_for("google_callback", _external=True)
    try:
        token_response = requests.post(
            "https://oauth2.googleapis.com/token",
            data={
                "code": code,
                "client_id": GOOGLE_CLIENT_ID,
                "client_secret": GOOGLE_CLIENT_SECRET,
                "redirect_uri": redirect_uri,
                "grant_type": "authorization_code",
            },
            timeout=15,
        )
        token_response.raise_for_status()
        access_token = token_response.json().get("access_token")
        info_response = requests.get(
            "https://openidconnect.googleapis.com/v1/userinfo",
            headers={"Authorization": f"Bearer {access_token}"},
            timeout=15,
        )
        info_response.raise_for_status()
        info = info_response.json()
        email = (info.get("email") or "").strip().lower()
        if not email or not info.get("email_verified", False):
            raise ValueError("O Google não confirmou o e-mail da conta.")
    except (requests.RequestException, ValueError, KeyError) as exc:
        app.logger.warning("Falha no login Google: %s", exc)
        flash("Não foi possível concluir o login com Google.", "danger")
        return redirect(url_for("login"))

    db = get_db()
    user = db.execute("SELECT * FROM usuarios WHERE lower(email)=?", (email,)).fetchone()
    nome = info.get("name") or email.split("@")[0]
    if user is None:
        base = "google_" + "".join(ch for ch in email.split("@")[0] if ch.isalnum())[:24]
        username = base or "google_usuario"
        counter = 1
        while db.execute("SELECT id FROM usuarios WHERE usuario=?", (username,)).fetchone():
            counter += 1
            username = f"{base}_{counter}"
        tipo = "admin" if email == ADMIN_EMAIL.lower() else "usuario"
        cur = db.execute(
            "INSERT INTO usuarios (nome,usuario,email,senha,tipo,ativo) VALUES (?,?,?,?,?,1)",
            (nome, username, email, generate_password_hash(secrets.token_urlsafe(32)), tipo),
        )
        db.commit()
        user = db.execute("SELECT * FROM usuarios WHERE id=?", (cur.lastrowid,)).fetchone()
    elif email == ADMIN_EMAIL.lower() and user["tipo"] != "admin":
        db.execute("UPDATE usuarios SET tipo='admin', ativo=1 WHERE id=?", (user["id"],))
        db.commit()
        user = db.execute("SELECT * FROM usuarios WHERE id=?", (user["id"],)).fetchone()

    if not user["ativo"]:
        flash("Esta conta está desativada.", "danger")
        return redirect(url_for("login"))
    session.clear()
    session["usuario_id"] = user["id"]
    flash("Login com Google realizado com sucesso.", "success")
    return redirect(url_for("home"))


@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        user = get_db().execute("SELECT * FROM usuarios WHERE usuario=?", (request.form.get("usuario", "").strip().lower(),)).fetchone()
        if user and user["ativo"] and check_password_hash(user["senha"], request.form.get("senha", "")):
            session.clear(); session["usuario_id"] = user["id"]
            return redirect(request.args.get("next") or url_for("home"))
        flash("Usuário ou senha inválidos.", "danger")
    body = """<h1>Entrar</h1><form class='panel' method='post'><div class='field'><label>Usuário</label><input name='usuario' required></div><br><div class='field'><label>Senha</label><input type='password' name='senha' required></div><br><button class='btn'>Entrar</button></form><div class='panel' style='margin-top:14px;text-align:center'><p>ou</p><a class='btn secondary' href='{{ url_for("google_login") }}'>Entrar com Google</a></div>"""
    return page("Entrar", body)


@app.route("/logout")
def logout():
    session.clear(); flash("Você saiu do sistema.", "success"); return redirect(url_for("home"))


@app.route("/favoritos")
@login_required
def favoritos():
    rows = get_db().execute("""SELECT f.* FROM farmacias f JOIN favoritos x ON x.farmacia_id=f.id WHERE x.usuario_id=? ORDER BY f.nome""", (session["usuario_id"],)).fetchall()
    body = """<h1>Meus favoritos</h1>{% if rows %}<div class='grid'>{% for f in rows %}<article class='card'><h3>{{ f['nome'] }}</h3><p class='info'>📍 {{ f['endereco'] or 'Endereço não informado' }}<br>☎️ {{ f['telefone'] or 'Telefone não informado' }}</p><form method='post' action='{{ url_for("toggle_favorite", pharmacy_id=f["id"]) }}'><button class='btn danger'>Remover dos favoritos</button></form></article>{% endfor %}</div>{% else %}<div class='empty'>Você ainda não favoritou nenhuma farmácia.</div>{% endif %}"""
    return page("Favoritos", body, rows=rows)


@app.route("/favoritos/<int:pharmacy_id>", methods=["POST"])
@login_required
def toggle_favorite(pharmacy_id):
    db = get_db()
    exists = db.execute("SELECT id FROM favoritos WHERE usuario_id=? AND farmacia_id=?", (session["usuario_id"], pharmacy_id)).fetchone()
    if exists: db.execute("DELETE FROM favoritos WHERE id=?", (exists["id"],)); flash("Farmácia removida dos favoritos.", "success")
    else: db.execute("INSERT OR IGNORE INTO favoritos (usuario_id,farmacia_id) VALUES (?,?)", (session["usuario_id"], pharmacy_id)); flash("Farmácia adicionada aos favoritos.", "success")
    db.commit(); return redirect(request.referrer or url_for("home"))


# -----------------------------------------------------------------------------
# Administração
# -----------------------------------------------------------------------------

@app.route("/admin", methods=["GET", "POST"])
@admin_required
def admin():
    db = get_db()
    if request.method == "POST":
        f = request.form
        nome = f.get("nome", "").strip()
        if not nome:
            flash("O nome da farmácia é obrigatório.", "danger")
        else:
            try: avaliacao = max(0, min(5, float(f.get("avaliacao", "0"))))
            except ValueError: avaliacao = 0
            db.execute("""INSERT INTO farmacias (nome,categoria,endereco,bairro,telefone,horarios,avaliacao,entrega,destaque,latitude,longitude,descricao) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""", (nome, f.get("categoria", "Farmácia"), f.get("endereco", "").strip(), f.get("bairro", "").strip(), f.get("telefone", "").strip(), f.get("horarios", "").strip(), avaliacao, f.get("entrega", "Não informado"), f.get("destaque", "Não"), f.get("latitude", "").strip(), f.get("longitude", "").strip(), f.get("descricao", "").strip()))
            db.commit(); flash("Farmácia cadastrada.", "success")
        return redirect(url_for("admin"))
    pharmacies = db.execute("SELECT * FROM farmacias ORDER BY id DESC").fetchall()
    people = db.execute("SELECT p.*,u.usuario,u.ativo FROM pessoas_publicas p JOIN usuarios u ON u.id=p.usuario_id ORDER BY p.id DESC").fetchall()
    body = """<h1>Administração</h1><p class='muted'>Área exclusiva do administrador. Esta página não aparece no cadastro público.</p><div class='tabs'><a class='tab active' href='#farmacias'>Cadastro de Farmácia</a><a class='tab' href='#publicos'>Cadastros Públicos</a></div><div id='farmacias'></div><form class='panel' method='post'><h2>Adicionar farmácia</h2><div class='form-grid'><div class='field'><label>Nome *</label><input name='nome' required></div><div class='field'><label>Categoria</label><select name='categoria'><option>Farmácia</option><option>Drogaria</option><option>Farmácia de manipulação</option></select></div><div class='field'><label>Endereço</label><input name='endereco'></div><div class='field'><label>Bairro</label><input name='bairro'></div><div class='field'><label>Telefone</label><input name='telefone'></div><div class='field'><label>Horários</label><input name='horarios' placeholder='Seg-Sex 08:00-18:00'></div><div class='field'><label>Avaliação</label><input name='avaliacao' type='number' min='0' max='5' step='0.1' value='0'></div><div class='field'><label>Entrega</label><select name='entrega'><option>Não informado</option><option>Sim</option><option>Não</option></select></div><div class='field'><label>Destaque</label><select name='destaque'><option>Não</option><option>Sim</option></select></div><div class='field'><label>Latitude</label><input name='latitude'></div><div class='field'><label>Longitude</label><input name='longitude'></div><div class='field full'><label>Descrição</label><textarea name='descricao'></textarea></div></div><button class='btn'>+ Cadastrar farmácia</button></form><h2>Farmácias cadastradas</h2>{% for f in pharmacies %}<div class='card'><h3>{{ f['nome'] }}</h3><p class='info'>{{ f['categoria'] }} · {{ f['endereco'] or 'Sem endereço' }} · Nota {{ '%.1f'|format(f['avaliacao']) }}</p><form method='post' action='{{ url_for("delete_pharmacy", pharmacy_id=f["id"]) }}' onsubmit='return confirm("Excluir esta farmácia?")'><button class='btn danger'>Excluir</button></form></div>{% else %}<p>Nenhuma farmácia cadastrada.</p>{% endfor %}<div id='publicos'></div><h2>Cadastros Públicos / Pessoas públicas pendentes</h2>{% for p in people %}<div class='card'><h3>{{ p['nome_publico'] or p['nome_completo'] }}</h3><p class='info'>{{ p['cargo'] or 'Sem cargo' }} · Status: {{ p['status'] }}</p>{% if p['status'] == 'Pendente' %}<div class='actions'><form method='post' action='{{ url_for("set_public_status", person_id=p["id"], status="Aprovado") }}'><button class='btn'>Aprovar</button></form><form method='post' action='{{ url_for("set_public_status", person_id=p["id"], status="Rejeitado") }}'><button class='btn danger'>Rejeitar</button></form></div>{% endif %}</div>{% else %}<p>Nenhum cadastro de pessoa pública.</p>{% endfor %}"""
    return page("Administração", body, pharmacies=pharmacies, people=people)


@app.route("/admin/farmacias/<int:pharmacy_id>/excluir", methods=["POST"])
@admin_required
def delete_pharmacy(pharmacy_id):
    db = get_db(); db.execute("DELETE FROM farmacias WHERE id=?", (pharmacy_id,)); db.commit(); flash("Farmácia excluída.", "success"); return redirect(url_for("admin"))


@app.route("/admin/pessoas/<int:person_id>/<status>", methods=["POST"])
@admin_required
def set_public_status(person_id, status):
    if status not in ("Aprovado", "Rejeitado"): return redirect(url_for("admin"))
    db = get_db(); db.execute("UPDATE pessoas_publicas SET status=? WHERE id=?", (status, person_id)); db.commit(); flash(f"Cadastro marcado como {status.lower()}.", "success"); return redirect(url_for("admin"))


# -----------------------------------------------------------------------------
# Inicialização
# -----------------------------------------------------------------------------

with app.app_context():
    init_db()


if __name__ == "__main__":
    # Importação local para evitar erro mesmo se o Colab tiver executado
    # somente parte dos imports da célula.
    import threading
    import time

    PORTA = int(os.environ.get("PORT", "5000"))
    print("Farma Tech iniciado SOMENTE localmente. Não há publicação externa.")
    print("Acesso: http://127.0.0.1:5000")
    print("Login administrativo inicial: admin / 123456")

    servidor = threading.Thread(
        target=lambda: app.run(
            host="127.0.0.1",
            port=PORTA,
            debug=False,
            use_reloader=False,
        ),
        daemon=True,
    )
    servidor.start()
    time.sleep(2)

    # Abre apenas uma janela interna do próprio Colab; não usa ngrok,
    # não cria URL pública e não publica o sistema na internet.
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(PORTA)
    except Exception:
        print("Se estiver no computador, abra http://127.0.0.1:5000")


Farma Tech iniciado SOMENTE localmente. Não há publicação externa.
Acesso: http://127.0.0.1:5000
Login administrativo inicial: admin / 123456
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>